# 07b — Noisy simulation on ibm_pittsburgh

Simulates the **exact circuits that ran on the QPU** (loaded from `.qpy`) on the noise model of ibm_pittsburgh.
Saves after every circuit to `results/sim_noisy_pittsburgh.json`: it can be stopped and restarted at any time,
and it continues where it stopped. Local and free (no QPU time).

Limitation: `AerSimulator.from_backend` uses the calibration at simulation time, not the one at submission (T15-02).

In [2]:
import json
import os
import time

import pandas as pd
from qiskit import qpy
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
noisy_simulator = AerSimulator.from_backend(service.backend("ibm_pittsburgh"))
manifest = pd.read_csv("../data/qpu/manifest.csv")
print(noisy_simulator)

AerSimulator('aer_simulator_from(ibm_pittsburgh)'
             noise_model=<NoiseModel on ['id', 'xslow', 'reset_2', 'reset', 'x', 'measure', 'measure_reset', 'measure_reset_2', 'sx', 'measure_2', 'cz']>)


### Step A: how long does one large circuit take?

In [3]:
job_row = manifest[manifest["tag"] == "main_n14"].iloc[0]
with open(job_row["circuits"], "rb") as f:
    circuits = qpy.load(f)
with open(job_row["raw"]) as f:
    labels = json.load(f)["labels"]

i = labels.index("ring_14|bisection|p2")          # 584 CZ, the heaviest non-stress circuit
for shots in [250, 1000]:
    start = time.time()
    noisy_simulator.run(circuits[i], shots=shots, seed_simulator=42).result()
    print(shots, "shots:", round(time.time() - start, 1), "s")

250 shots: 7.8 s
1000 shots: 30.9 s


### Step B: all 38 circuits (choose SHOTS after step A)

In [4]:
NOISY_PATH = "../results/sim_noisy_pittsburgh.json"
SHOTS = 4000

# start from what was already simulated (if anything)
if os.path.exists(NOISY_PATH):
    with open(NOISY_PATH) as f:
        noisy_counts = json.load(f)
else:
    noisy_counts = {}
print("already done:", len(noisy_counts), "circuits")

for _, job_row in manifest[manifest["tag"].str.startswith("main")].iterrows():
    with open(job_row["circuits"], "rb") as f:
        circuits = qpy.load(f)
    with open(job_row["raw"]) as f:
        labels = json.load(f)["labels"]

    for label, circuit in zip(labels, circuits):
        if label in noisy_counts:
            continue                                   # simulated in an earlier run
        start = time.time()
        result = noisy_simulator.run(circuit, shots=SHOTS, seed_simulator=42).result()
        noisy_counts[label] = result.get_counts()

        with open(NOISY_PATH, "w") as f:               # save after EVERY circuit
            json.dump(noisy_counts, f)
        print(label, "done in", round(time.time() - start), "s")

print("total:", len(noisy_counts), "circuits")

already done: 0 circuits
random_6|bisection|p1 done in 1 s
random_6|bisection|p2 done in 1 s
random_6|maxcut|p1 done in 1 s
random_6|maxcut|p2 done in 1 s
regular3_6|bisection|p1 done in 1 s
regular3_6|bisection|p2 done in 1 s
regular3_6|maxcut|p1 done in 1 s
regular3_6|maxcut|p2 done in 1 s
ring_6|bisection|p1 done in 1 s
ring_6|bisection|p2 done in 1 s
ring_6|maxcut|p1 done in 1 s
ring_6|maxcut|p2 done in 1 s
random_10|bisection|p1 done in 1 s
random_10|bisection|p2 done in 1 s
random_10|maxcut|p1 done in 1 s
random_10|maxcut|p2 done in 1 s
regular3_10|bisection|p1 done in 1 s
regular3_10|bisection|p2 done in 1 s
regular3_10|maxcut|p1 done in 1 s
regular3_10|maxcut|p2 done in 1 s
ring_10|bisection|p1 done in 1 s
ring_10|bisection|p2 done in 1 s
ring_10|maxcut|p1 done in 4 s
ring_10|maxcut|p2 done in 4 s
random_14|bisection|p1 done in 61 s
random_14|bisection|p2 done in 131 s
random_14|maxcut|p1 done in 56 s
random_14|maxcut|p2 done in 123 s
regular3_14|bisection|p1 done in 81 s
regul